# 16 rank mimicking yolo26s to yolo26n — Google Colab

1. Set **Runtime > Change runtime type > GPU** for training.
2. Edit `DATASET_ZIP` and `DRIVE_OUTPUT_ROOT` below.
3. Add `WANDB_API_KEY` in **Colab Secrets** and enable notebook access (training only).
4. Run All. Research hyperparameters and seed schedules are in the next configuration cell.

Source: `14_rm_yolo26n_teacher_yolo26s.ipynb`. Experiment IDs/artifact names are preserved. This copy is standalone; no repository checkout is required.

### Method notes

Poltrada V3 · seed 42 · Colab GPU.

Tambahkan dataset, aktifkan GPU/Internet, dan secret `WANDB_API_KEY`. Run All langsung menjalankan training **100 epoch** pada 640/batch 16. `RUN_FULL_TRAINING=True`; smoke tambahan dinonaktifkan dengan `RUN_SMOKE_TEST=False`. Audit teacher dan verifikasi numerik tetap berjalan.

RM memindahkan ranking kandidat per objek: softmax pada sumbu kandidat, lalu KL teacher ke student. Dua indikatornya adalah skor kelas GT dan IoU box dengan GT; mean antar-instance, bobot RM 4. Kandidat adalah cell P3/P4/P5 sebelum NMS, dipilih teacher top-10 IoU di dalam GT dengan ownership tunggal. Skor student tetap terhubung ke autograd. Pemilihan kandidat adalah adaptasi untuk YOLO26; tidak ada PFI. [Paper RM](https://ojs.aaai.org/index.php/AAAI/article/download/20018/19777).


## Colab paths

Keep the ZIP unchanged in Drive. Extraction and the derived YAML are local; source images, labels and YAML are not edited. For multiple YAML files, set `DATASET_YAML_RELATIVE` to the desired archive-relative path.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import uuid

# Edit these two paths to match your Google Drive.
DATASET_ZIP = "/content/drive/MyDrive/Skripsi/datasets/poltrada-v3-yolo26-clean.zip"
DRIVE_OUTPUT_ROOT = "/content/drive/MyDrive/Skripsi/yolo26-kd-indotraffic/results"
DATASET_YAML_RELATIVE = ""  # Set only if the ZIP contains more than one YAML.
SOURCE_DATA_DECLARATION = '/kaggle/input/datasets/ilhamrafiqin/poltrada-v3-yolo26-clean/data.yaml'  # Original teacher provenance; not an active Colab path.
COLAB_RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ") + "_" + uuid.uuid4().hex[:6]
COLAB_LOCAL_DIR = Path("/content/yolo26_colab") / COLAB_RUN_ID
COLAB_OUTPUT_DIR = Path(DRIVE_OUTPUT_ROOT) / "16_rank_mimicking_yolo26s_to_yolo26n" / COLAB_RUN_ID
COLAB_DATA = str(COLAB_LOCAL_DIR / "runtime/data.yaml")


## Research configuration

Preserved from the original notebook. The trio defaults to 100 epochs with smoke disabled. Existing multi-seed queues remain unchanged; 100 epochs is per seed.

In [ ]:
EXPERIMENT_ID = "14_rm_yolo26n_teacher_yolo26s"
METHOD = "rm"
MODEL_WEIGHTS = "yolo26n.pt"
DATA = COLAB_DATA
EPOCHS = 100
IMGSZ = 640
BATCH = 16
SEED = 42
DEVICE = 0
WORKERS = 2
OPTIMIZER = "AdamW"
LR = 0.001
WEIGHT_DECAY = 0.0005
WARMUP_EPOCHS = 3.0
CLOSE_MOSAIC = 10
AMP = True
EVAL_SPLIT = "val"
PROJECT = str(COLAB_OUTPUT_DIR)
RUN_FULL_TRAINING = True  # Training penelitian sesuai EPOCHS.
RUN_SMOKE_TEST = False  # True hanya jika ingin smoke test tambahan 1 epoch.
METHOD_KWARGS = {'rm_weight': 4.0, 'rm_cls_weight': 1.0, 'rm_iou_weight': 1.0, 'candidate_topk': 10, 'candidate_iou_threshold': 0.0}
WANDB_ENTITY = "ilhamrafiqin-brawijaya-university"
WANDB_PROJECT = "yolom26-kd-indotraffic"
WANDB_GROUP = "T26s_S26n"
TEACHER_ARTIFACT = "ilhamrafiqin-brawijaya-university/yolom26-kd-indotraffic/02_baseline_yolo26s-q5kp9k9d-checkpoints:v62"
TEACHER_SHA256 = "5bc3ae61f8da1c9f03916922de10146805c27c64a11e46edd766a00d0f6f18fd"


## Colab: Drive, dependencies, and local dataset

Train from the local extracted ZIP; outputs go to Drive and existing W&B callbacks. Authorize Drive mounting when prompted.

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")
COLAB_LOCAL_DIR.mkdir(parents=True, exist_ok=False)
COLAB_OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
os.chdir(COLAB_LOCAL_DIR)  # Downloads and temporary files stay on the Colab VM.
os.environ["WANDB_CACHE_DIR"] = str(COLAB_LOCAL_DIR / "wandb-cache")
os.environ["WANDB_DIR"] = str(COLAB_LOCAL_DIR)
os.environ["PIP_CACHE_DIR"] = str(COLAB_LOCAL_DIR / "pip-cache")
os.environ["TORCH_HOME"] = str(COLAB_LOCAL_DIR / "torch-cache")
print("Persistent output:", COLAB_OUTPUT_DIR)


In [ ]:
from importlib import metadata
import subprocess
import sys

# Preserve Colab's installed PyTorch/torchvision build during dependency resolution.
constraints = []
for package in ("torch", "torchvision"):
    try:
        constraints.append(package + "==" + metadata.version(package))
    except metadata.PackageNotFoundError:
        if True:
            raise RuntimeError("Use a Colab GPU runtime with PyTorch installed.") from None
constraint_file = COLAB_LOCAL_DIR / "colab-torch-constraints.txt"
constraint_file.write_text("\n".join(constraints), encoding="utf-8")
for distribution, package, version in [('ultralytics', 'ultralytics', '8.4.155'), ('wandb', 'wandb', '0.28.1'), ('PyYAML', 'PyYAML', None)]:
    try:
        installed = metadata.version(distribution)
    except metadata.PackageNotFoundError:
        installed = None
    if installed is None or (version is not None and installed != version):
        requirement = package if version is None else package + "==" + version
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-c", str(constraint_file), requirement])


In [ ]:
"""Portable helpers embedded into each generated Colab notebook (no repo imports)."""

import hashlib
import os
import shutil
import stat
import zipfile
from pathlib import Path, PurePosixPath

import yaml


def colab_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def colab_api_key(name="WANDB_API_KEY"):
    """Prefer an existing environment value; otherwise request a Colab Secret."""
    value = os.environ.get(name, "").strip()
    if not value:
        try:
            from google.colab import userdata

            value = userdata.get(name)
        except Exception:  # noqa: BLE001 -- Normalize UI failures without exposing secret-bearing payloads.
            raise RuntimeError(
                f"Add {name} in Colab Secrets (key icon), enable notebook access, "
                "then rerun this cell. Never paste the key into notebook code."
            ) from None
    if not isinstance(value, str) or not value.strip():
        raise RuntimeError(f"Colab Secret {name} is empty; update it and rerun this cell.")
    os.environ[name] = value.strip()
    return value.strip()


def colab_require_gpu(torch_module):
    if not torch_module.cuda.is_available():
        raise RuntimeError("Select Runtime > Change runtime type > GPU in Colab, then Run All.")


def colab_extract_zip(archive, destination):
    """Reject unsafe/duplicate members before extracting; ZIP CRCs validate reads."""
    destination = Path(destination).resolve()
    with zipfile.ZipFile(archive) as source:
        members = source.infolist()
        targets = []
        seen = set()
        for member in members:
            name = member.filename.replace("\\", "/")
            relative = PurePosixPath(name)
            if member.is_dir() and str(relative) == ".":
                continue  # Common `zip -r archive.zip .` root-directory entry.
            if (relative.is_absolute() or not relative.parts or ".." in relative.parts
                    or any(":" in part for part in relative.parts)
                    or stat.S_ISLNK(member.external_attr >> 16)):
                raise ValueError(f"Unsafe ZIP member: {member.filename!r}")
            target = destination.joinpath(*relative.parts).resolve()
            if not target.is_relative_to(destination) or target in seen:
                raise ValueError(f"Unsafe or duplicate ZIP member: {member.filename!r}")
            seen.add(target)
            targets.append((member, target))
        needed = sum(member.file_size for member in members)
        if needed > shutil.disk_usage(destination.parent).free:
            raise OSError("Insufficient Colab local disk for the extracted dataset.")
        destination.mkdir(parents=True, exist_ok=False)
        for member, target in targets:
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with source.open(member) as incoming, target.open("wb") as outgoing:
                    shutil.copyfileobj(incoming, outgoing)
        return {"files": sum(not member.is_dir() for member in members), "bytes": needed}


def colab_resolve_reference(value, root, extracted, declared_root, source_declaration):
    """Relocate the declared root, keeping directory names and split membership."""
    value = str(value).replace("\\", "/")
    relative = PurePosixPath(value)
    if relative.is_absolute():
        prefixes = [str(declared_root or ""), str(PurePosixPath(source_declaration).parent)]
        for prefix in prefixes:
            prefix_path = PurePosixPath(prefix)
            if prefix and prefix_path.is_absolute() and relative.is_relative_to(prefix_path):
                relative = relative.relative_to(prefix_path)
                break
        else:
            raise ValueError(f"Cannot relocate absolute dataset path: {value}")
    candidate = root.joinpath(*relative.parts).resolve()
    # Same leading ../ compatibility used by the pinned Ultralytics loader.
    if not candidate.exists() and value.startswith("../"):
        candidate = (root / value[3:]).resolve()
    if not candidate.is_relative_to(extracted) or not candidate.exists():
        raise FileNotFoundError(f"Dataset path is missing or outside the extracted ZIP: {value}")
    return candidate


def prepare_colab_dataset(zip_path, local_directory, yaml_relative, source_declaration):
    """Copy a Drive ZIP locally and write a separate, fully resolved runtime YAML."""
    zip_path, local = Path(zip_path), Path(local_directory).resolve()
    if not zip_path.is_file():
        raise FileNotFoundError(f"Dataset ZIP not found: {zip_path}. Edit DATASET_ZIP in Colab configuration.")
    local.mkdir(parents=True, exist_ok=True)
    runtime_yaml = local / "runtime/data.yaml"
    # A completed setup is immutable within this run; do not overwrite/reuse partial extraction.
    if (local / "source").exists() or runtime_yaml.exists():
        raise RuntimeError("Dataset setup already started for this run. Rerun the Colab configuration cell to create a new run directory.")
    archive = local / "dataset.zip"
    if zip_path.stat().st_size > shutil.disk_usage(local).free:
        raise OSError("Insufficient Colab local disk to copy DATASET_ZIP.")
    shutil.copyfile(zip_path, archive)
    archive_sha = colab_sha256(archive)
    if colab_sha256(zip_path) != archive_sha:
        raise ValueError("Dataset ZIP changed or was corrupted during the copy from Drive.")
    extracted = local / "source"
    inventory = colab_extract_zip(archive, extracted)
    if yaml_relative:
        original_yaml = (extracted / yaml_relative).resolve()
        if not original_yaml.is_relative_to(extracted) or not original_yaml.is_file():
            raise FileNotFoundError("DATASET_YAML_RELATIVE must name a YAML file inside the ZIP.")
    else:
        candidates = sorted(p for p in extracted.rglob("*") if p.suffix.lower() in {".yaml", ".yml"})
        if len(candidates) != 1:
            choices = [str(p.relative_to(extracted)) for p in candidates]
            raise ValueError(f"Expected one dataset YAML, found {choices}. Set DATASET_YAML_RELATIVE explicitly.")
        original_yaml = candidates[0]
    class UniqueDatasetLoader(yaml.SafeLoader):
        pass

    def unique_mapping(loader, node):
        mapping = {}
        for key_node, value_node in node.value:
            key = loader.construct_object(key_node)
            if key in mapping:
                raise ValueError(f"Duplicate dataset YAML key: {key!r}; fix the source ZIP explicitly.")
            mapping[key] = loader.construct_object(value_node)
        return mapping

    UniqueDatasetLoader.add_constructor(yaml.resolver.BaseResolver.DEFAULT_MAPPING_TAG, unique_mapping)
    raw = yaml.load(original_yaml.read_text(encoding="utf-8-sig"), Loader=UniqueDatasetLoader)
    if not isinstance(raw, dict) or not raw.get("train") or not raw.get("val"):
        raise ValueError("Dataset YAML must declare train, val and class names.")
    names = raw.get("names")
    if isinstance(names, dict):
        keys = sorted(names, key=lambda key: int(key))
        if [int(key) for key in keys] != list(range(len(keys))):
            raise ValueError("Class IDs must be consecutive starting at zero.")
        ordered_names = [str(names[key]) for key in keys]
    elif isinstance(names, list):
        ordered_names = [str(name) for name in names]
    else:
        raise TypeError("Dataset YAML names must be a list or a class-ID mapping.")
    if not ordered_names or int(raw.get("nc", len(ordered_names))) != len(ordered_names):
        raise ValueError("Dataset nc and class names disagree.")
    declared_root = raw.get("path")
    root = original_yaml.parent
    if declared_root and not PurePosixPath(str(declared_root).replace("\\", "/")).is_absolute():
        root = (root / declared_root).resolve()
    if not root.is_relative_to(extracted) or not root.is_dir():
        raise ValueError("Dataset YAML root must resolve inside the extracted ZIP.")
    relocated = dict(raw)
    relocated["path"] = str(root)
    runtime_yaml.parent.mkdir(parents=True, exist_ok=True)
    split_counts = {}
    # Match Ultralytics 8.4.155 IMG_FORMATS, including its less common formats.
    image_suffixes = {".avif", ".bmp", ".dng", ".heic", ".heif", ".jp2", ".jpeg", ".jpg", ".mpo", ".png", ".tif", ".tiff", ".webp"}
    for split in ("train", "val", "test"):
        if not raw.get(split):
            continue
        values = raw[split] if isinstance(raw[split], list) else [raw[split]]
        outputs, count = [], 0
        for index, value in enumerate(values):
            if not isinstance(value, str):
                raise TypeError(f"Invalid {split} path: {value!r}")
            path = colab_resolve_reference(value, root, extracted, declared_root, source_declaration)
            if path.is_dir():
                count += sum(p.is_file() and p.suffix.lower() in image_suffixes for p in path.rglob("*"))
                outputs.append(str(path))
            elif path.suffix.lower() == ".txt":
                entries = []
                for line in path.read_text(encoding="utf-8-sig").splitlines():
                    if not line.strip():
                        continue
                    image_root = path.parent if line.strip().startswith("./") else root
                    image = colab_resolve_reference(line.strip(), image_root, extracted, declared_root, source_declaration)
                    if not image.is_file() or image.suffix.lower() not in image_suffixes:
                        raise ValueError(f"Split list references a non-image: {line}")
                    entries.append(str(image))
                derived = runtime_yaml.parent / f"{split}-{index}.txt"
                derived.write_text("\n".join(entries) + "\n", encoding="utf-8")
                outputs.append(str(derived))
                count += len(entries)
            else:
                raise ValueError(f"Split must reference an image directory or TXT list: {value}")
        if count == 0:
            raise ValueError(f"No images found in {split}; check the ZIP and YAML paths.")
        relocated[split] = outputs if isinstance(raw[split], list) else outputs[0]
        split_counts[split] = count
    runtime_yaml.write_text(yaml.safe_dump(relocated, sort_keys=False, allow_unicode=True), encoding="utf-8")
    provenance = {
        "runtime": "google-colab", "drive_zip": str(zip_path), "zip_sha256": archive_sha,
        "extracted_inventory": inventory, "original_yaml": str(original_yaml),
        "original_yaml_sha256": colab_sha256(original_yaml), "runtime_yaml": str(runtime_yaml),
        "runtime_yaml_sha256": colab_sha256(runtime_yaml), "source_dataset_declaration": source_declaration,
        "class_names": ordered_names, "split_image_counts": split_counts,
        "relocation_only": True,
        "identity_note": "ZIP provenance is recorded; equality with the original Kaggle image/label bytes is not independently established.",
    }
    return str(runtime_yaml), provenance


In [ ]:
import json
import torch
colab_require_gpu(torch)
COLAB_DATA, COLAB_DATA_PROVENANCE = prepare_colab_dataset(
    DATASET_ZIP, COLAB_LOCAL_DIR, DATASET_YAML_RELATIVE, SOURCE_DATA_DECLARATION)
DATA = COLAB_DATA
(COLAB_OUTPUT_DIR / "dataset_provenance.json").write_text(
    json.dumps(COLAB_DATA_PROVENANCE, indent=2, ensure_ascii=False), encoding="utf-8")
print("Dataset:", COLAB_DATA)
print("Split image counts:", COLAB_DATA_PROVENANCE["split_image_counts"])


## 2. Persiapan Colab
PyTorch GPU bawaan Colab tetap digunakan.

In [ ]:
import importlib.metadata as metadata
import subprocess
import sys

for package, version in (("ultralytics", "8.4.155"), ("wandb", "0.28.1")):
    try:
        installed = metadata.version(package)
    except metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", f"{package}=={version}"])

import copy
import hashlib
import json
import math
import os
import shutil
import tempfile
import time
import uuid
from datetime import datetime, timezone
from pathlib import Path

import torch
from torch import nn
from ultralytics import YOLO, settings
from ultralytics.data.utils import check_det_dataset
from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils.torch_utils import ModelEMA, unwrap_model
import wandb

if metadata.version("ultralytics") != "8.4.155":
    raise RuntimeError("Restart kernel setelah instalasi Ultralytics.")


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("Aktifkan GPU di Colab: Runtime > Change runtime type > GPU.")
settings.update({"wandb": False})  # Logging diatur oleh callback notebook.
os.environ["WANDB_API_KEY"] = colab_api_key()
wandb.login(key=os.environ["WANDB_API_KEY"])
if wandb.run is not None:
    raise RuntimeError("Selesaikan run W&B yang aktif sebelum menjalankan notebook ini.")

SESSION_DIR = Path(PROJECT) / f"{EXPERIMENT_ID}_{datetime.now():%Y%m%d_%H%M%S}_{uuid.uuid4().hex[:6]}"
SESSION_DIR.mkdir(parents=True, exist_ok=False)
print("GPU:", torch.cuda.get_device_name(DEVICE))
print("Output:", SESSION_DIR)


## 3. Dataset dan teacher
Split dan anotasi tetap. Teacher dikunci ke `best.pt` artifact v62; metadata epoch-nya 63, bukan 62.

In [ ]:
def class_names(names):
    return [str(names[k]) for k in sorted(names, key=lambda k: int(k))] if isinstance(names, dict) else list(names)

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

data_info = check_det_dataset(DATA, autodownload=False)
if EVAL_SPLIT != "val" or not data_info.get("train") or not data_info.get("val"):
    raise ValueError("Gunakan split train dan val; test tidak dipakai untuk pemilihan checkpoint.")
DATASET_NAMES = class_names(data_info["names"])
DATASET_INFO = {"data": DATA, "names": DATASET_NAMES, "yaml_sha256": file_sha256(DATA)}
print("Kelas:", DATASET_NAMES)


In [ ]:
artifact = wandb.Api().artifact(TEACHER_ARTIFACT, type="model")
producer = artifact.logged_by()
if artifact.version != "v62" or producer is None or list(producer.path) != [WANDB_ENTITY, WANDB_PROJECT, "q5kp9k9d"]:
    raise ValueError("Artifact bukan teacher baseline YOLO26s yang ditetapkan.")
TEACHER_PATH = Path(artifact.get_path("best.pt").download(root=str(COLAB_LOCAL_DIR / "teacher")))
if file_sha256(TEACHER_PATH) != TEACHER_SHA256:
    raise ValueError("SHA-256 teacher berbeda; tidak ada fallback ke pretrained teacher.")
teacher = YOLO(str(TEACHER_PATH)).model
reference = DetectionModel("yolo26s.yaml", nc=data_info["nc"], verbose=False)
if teacher.yaml.get("scale") != "s" or class_names(teacher.names) != DATASET_NAMES:
    raise ValueError("Arsitektur atau urutan kelas teacher berbeda dari dataset.")
if {k: tuple(v.shape) for k, v in teacher.state_dict().items()} != {k: tuple(v.shape) for k, v in reference.state_dict().items()}:
    raise ValueError("Schema checkpoint bukan YOLO26s yang sesuai.")
TEACHER_INFO = {"artifact": TEACHER_ARTIFACT, "sha256": TEACHER_SHA256,
                "names": DATASET_NAMES, "artifact_metadata": dict(artifact.metadata)}
(SESSION_DIR / "teacher_provenance.json").write_text(json.dumps(TEACHER_INFO, indent=2))
print("Teacher:", artifact.name, "| epoch metadata:", artifact.metadata.get("epoch"))
del teacher, reference


## 4. Loss dan model
Detection loss resmi YOLO26 tetap digunakan pada kedua cabang. KD ditambahkan sekali dengan faktor batch. Modul tambahan HEAD hanya untuk training.

In [ ]:
"""RM-only adaptation for YOLO26; notebook-embeddable, without repository imports.

Li et al., AAAI 2022 equations (5), (6), (10) and Table 4 are the source.
The positive-cell assignment and score-domain choices are explicit adaptations;
no author-verified implementation of this paper was located. No PFI is used.
"""

import math

import torch
from torch import nn
from ultralytics.utils.tal import dist2bbox, make_anchors


def rm_rank_kl(student_scores, teacher_scores):
    """Equation (6), one instance: softmax OVER CANDIDATES, then sum KL.

    Inputs are scalar classification probabilities or IoUs for matched cells.
    There is no class-axis softmax, temperature, sorting, or division by N.
    Teacher values are constants, while indexing student values retains autograd.
    Only the tiny per-instance vectors use FP64 to avoid cancellation when
    sigmoid scores are nearly equal; outputs return to FP32 under AMP.
    """
    if student_scores.ndim != 1 or student_scores.shape != teacher_scores.shape:
        raise ValueError("RM expects matching one-dimensional candidate score vectors")
    output_dtype = torch.float64 if student_scores.dtype == torch.float64 else torch.float32
    student_scores = student_scores.to(torch.float64)
    teacher_scores = teacher_scores.detach().to(device=student_scores.device, dtype=torch.float64)
    if student_scores.numel() < 2:
        return (student_scores.sum() * 0).to(output_dtype)
    student_log = student_scores.log_softmax(dim=0)
    teacher_log = teacher_scores.log_softmax(dim=0)
    return (teacher_log.exp() * (teacher_log - student_log)).sum().clamp(min=0).to(output_dtype)


def rm_box_iou(boxes, target):
    """Differentiable ordinary IoU of N xyxy boxes with one xyxy GT box.

    Native YOLO26 reg_max=1 distances may produce degenerate early predictions;
    nonpositive areas/intersections are clamped to zero, without detaching boxes.
    """
    intersection = (torch.minimum(boxes[:, 2:], target[2:]) -
                    torch.maximum(boxes[:, :2], target[:2])).clamp(min=0).prod(dim=-1)
    area = (boxes[:, 2:] - boxes[:, :2]).clamp(min=0).prod(dim=-1)
    target_area = (target[2:] - target[:2]).clamp(min=0).prod()
    return intersection / (area + target_area - intersection).clamp(min=1e-9)


@torch.no_grad()
def rm_candidate_groups(teacher_boxes, anchor_points, targets_xyxy, topk=10, iou_threshold=0.0):
    """Adapt positive-anchor assignment to common YOLO26 P3/P4/P5 cell IDs.

    A valid cell center is strictly inside GT and teacher decoded IoU > threshold.
    Each instance proposes its top-k IoUs over ALL pyramid levels, before NMS.
    Conflicts go to the largest teacher IoU; equal IoU uses first GT row. Within
    a top-k tie, stable cell order is used (P3, P4, P5, each row-major). Candidate
    membership never depends on student predictions. Empty groups are retained.
    This replaces detector-specific positive assignment, not the paper's KL.
    """
    if topk < 1 or not 0 <= iou_threshold < 1:
        raise ValueError("Candidate topk must be positive and IoU threshold in [0,1)")
    proposals = []
    qualities = []
    for target in targets_xyxy:
        inside = ((anchor_points > target[:2]) & (anchor_points < target[2:])).all(dim=1)
        quality = rm_box_iou(teacher_boxes, target)
        eligible = torch.nonzero(inside & (quality > iou_threshold), as_tuple=False).flatten()
        ranking = torch.argsort(quality[eligible], descending=True, stable=True)
        proposals.append(eligible[ranking[:topk]])
        qualities.append(quality)
    if not proposals:
        return []
    # Dense M x A ownership is modest (e.g. 100 GT x 8400 cells), no pairwise A^2 tensor.
    competition = teacher_boxes.new_full((len(proposals), len(teacher_boxes)), -1)
    for instance, (indices, quality) in enumerate(zip(proposals, qualities)):
        competition[instance, indices] = quality[indices]
    owner = competition.argmax(dim=0)
    return [indices[owner[indices] == instance] for instance, indices in enumerate(proposals)]


def rm_dense_predictions(predictions, strides):
    """Return pre-NMS one-to-many scores/logits, pixel xyxy boxes and cell centers.

    YOLO26 one-to-many is the dense branch with multiple candidates per object;
    the one-to-one branch still receives its complete native detection loss.
    reg_max=1 uses direct ltrb distances, NOT a DFL probability distribution.
    """
    if isinstance(predictions, tuple):
        predictions = predictions[1]
    if not isinstance(predictions, dict) or "one2many" not in predictions:
        raise ValueError("RM requires unfused YOLO26 end-to-end raw branches")
    dense = predictions["one2many"]
    if dense["boxes"].shape[1] != 4:
        raise ValueError("RM adaptation requires YOLO26 reg_max=1")
    anchors, stride_tensor = make_anchors(dense["feats"], strides, 0.5)
    # FP32 geometry and score distribution under autocast; no student detach.
    boxes = dist2bbox(dense["boxes"].permute(0, 2, 1).float(), anchors.float(), xywh=False)
    return dense["scores"].float(), boxes * stride_tensor.float(), anchors.float() * stride_tensor.float()


def rm_instance_losses(student_logits, teacher_logits, student_boxes, teacher_boxes,
                       anchor_points, batch, topk=10, iou_threshold=0.0):
    """Mean of per-instance candidate-axis KL for class scores AND box IoUs.

    M is the number of GT instances in the batch, including empty/singleton
    groups whose rank loss is zero. This deliberately does not divide by N.
    Student sigmoid class scores and IoUs remain differentiable. Teacher and
    geometry-based membership are frozen. Class identity is GT-specific, so
    adjacent objects of the same class are still distinct ranking groups.
    """
    if student_logits.shape != teacher_logits.shape or student_boxes.shape != teacher_boxes.shape:
        raise ValueError("Teacher/student dense candidate grids must match")
    if student_logits.ndim != 3 or student_boxes.shape[:2] != (student_logits.shape[0], student_logits.shape[2]):
        raise ValueError("Expected scores BxCxA and decoded boxes BxAx4")
    zero = student_logits.sum() * 0 + student_boxes.sum() * 0
    cls_loss, iou_loss = zero, zero
    images = batch["batch_idx"].view(-1).to(device=student_boxes.device, dtype=torch.long)
    classes = batch["cls"].view(-1).to(device=student_boxes.device, dtype=torch.long)
    targets = batch["bboxes"].to(device=student_boxes.device, dtype=torch.float32)
    instance_count = len(targets)
    valid_groups = 0
    candidate_count = 0
    if instance_count == 0:
        return cls_loss, iou_loss, {"rm_instances": 0, "rm_valid_groups": 0, "rm_candidates": 0}
    if images.numel() != instance_count or classes.numel() != instance_count:
        raise ValueError("GT batch indices, classes and bboxes must have matching lengths")
    if (images < 0).any() or (images >= student_logits.shape[0]).any():
        raise ValueError("GT image index outside current batch")
    if (classes < 0).any() or (classes >= student_logits.shape[1]).any():
        raise ValueError("GT class index outside the model class axis")
    height, width = batch["img"].shape[-2:]
    centers, half_sizes = targets[:, :2], targets[:, 2:] / 2
    targets_xyxy = torch.cat((centers - half_sizes, centers + half_sizes), dim=1)
    targets_xyxy = targets_xyxy * targets.new_tensor([width, height, width, height])
    for image in range(student_logits.shape[0]):
        target_ids = torch.nonzero(images == image, as_tuple=False).flatten()
        image_targets = targets_xyxy[target_ids]
        groups = rm_candidate_groups(teacher_boxes[image].detach(), anchor_points,
                                     image_targets, topk=topk, iou_threshold=iou_threshold)
        for local_instance, indices in enumerate(groups):
            candidate_count += indices.numel()
            if indices.numel() < 2:
                continue
            valid_groups += 1
            category = classes[target_ids[local_instance]]
            student_score = student_logits[image, category, indices].sigmoid()
            teacher_score = teacher_logits[image, category, indices].detach().sigmoid()
            cls_loss = cls_loss + rm_rank_kl(student_score, teacher_score)
            target = image_targets[local_instance]
            student_iou = rm_box_iou(student_boxes[image, indices], target)
            teacher_iou = rm_box_iou(teacher_boxes[image, indices].detach(), target)
            iou_loss = iou_loss + rm_rank_kl(student_iou, teacher_iou)
    counts = {"rm_instances": instance_count, "rm_valid_groups": valid_groups,
              "rm_candidates": candidate_count}
    return cls_loss / instance_count, iou_loss / instance_count, counts


In [ ]:
class RMModel(nn.Module):
    """Training-only RM wrapper; tensor forward and saved inference model are student-only."""

    def __init__(self, student_model, teacher_model, rm_weight=4.0, rm_cls_weight=1.0,
                 rm_iou_weight=1.0, candidate_topk=10, candidate_iou_threshold=0.0):
        super().__init__()
        self.student_model = student_model
        self.teacher_model = teacher_model
        self.rm_weight = float(rm_weight)
        self.rm_cls_weight = float(rm_cls_weight)
        self.rm_iou_weight = float(rm_iou_weight)
        self.candidate_topk = int(candidate_topk)
        self.candidate_iou_threshold = float(candidate_iou_threshold)
        if min(self.rm_weight, self.rm_cls_weight, self.rm_iou_weight) < 0:
            raise ValueError("RM weights must be nonnegative")
        if self.candidate_topk < 1 or not 0 <= self.candidate_iou_threshold < 1:
            raise ValueError("Invalid RM candidate selection parameters")
        student_head, teacher_head = student_model.model[-1], teacher_model.model[-1]
        if student_head.nc != teacher_head.nc:
            raise ValueError("Teacher/student number of classes must match")
        if not torch.equal(student_head.stride.cpu(), teacher_head.stride.cpu()):
            raise ValueError("Teacher/student pyramid strides must match")
        if list(student_head.stride.cpu()) != [8, 16, 32]:
            raise ValueError("Expected YOLO26 P3/P4/P5 strides 8,16,32")
        if student_head.reg_max != 1 or teacher_head.reg_max != 1:
            raise ValueError("Expected YOLO26 direct regression with reg_max=1")
        if any(getattr(head, "one2one_cv2", None) is None or
               getattr(head, "one2one_cv3", None) is None for head in (student_head, teacher_head)):
            raise ValueError("Expected YOLO26 one-to-many and one-to-one branches")
        self.last_metrics = {}
        self._freeze_teacher()

    def __getattr__(self, name):
        # Delegate normal Ultralytics metadata/model access without registering
        # another copy of student modules in the wrapper or state dict.
        try:
            return super().__getattr__(name)
        except AttributeError:
            student = super().__getattr__("student_model")
            return getattr(student, name)

    def _freeze_teacher(self):
        if self.teacher_model is not None:
            self.teacher_model.requires_grad_(False)
            self.teacher_model.eval()

    def train(self, mode=True):
        super().train(mode)
        self._freeze_teacher()
        return self

    def forward(self, x, *args, **kwargs):
        if isinstance(x, dict):
            return self.loss(x, *args, **kwargs)
        return self.student_model(x, *args, **kwargs)

    def loss(self, batch, preds=None):
        if preds is None:
            preds = self.student_model(batch["img"])
        detection_vector, native_items = self.student_model.loss(batch, preds)
        zero = detection_vector.sum() * 0
        cls_loss, iou_loss = zero, zero
        counts = {"rm_instances": 0, "rm_valid_groups": 0, "rm_candidates": 0}
        if self.training:
            self._freeze_teacher()
            with torch.no_grad():
                teacher_preds = self.teacher_model(batch["img"])
                t_logits, t_boxes, t_anchors = rm_dense_predictions(teacher_preds, self.teacher_model.stride)
            s_logits, s_boxes, s_anchors = rm_dense_predictions(preds, self.student_model.stride)
            if not torch.equal(s_anchors, t_anchors):
                raise ValueError("Teacher/student cell centers differ: candidate pairing invalid")
            cls_loss, iou_loss, counts = rm_instance_losses(
                s_logits, t_logits, s_boxes, t_boxes, s_anchors, batch,
                topk=self.candidate_topk, iou_threshold=self.candidate_iou_threshold)
        rm_loss = self.rm_cls_weight * cls_loss + self.rm_iou_weight * iou_loss
        kd_loss = self.rm_weight * rm_loss
        batch_size = batch["img"].shape[0]
        # Native vector has ALREADY been batch-scaled; multiply KD exactly once.
        vector = torch.cat((detection_vector, (kd_loss * batch_size).reshape(1)))
        loss_items = dict(native_items)
        loss_items["dis_loss"] = kd_loss.detach()
        self.last_metrics = {
            "det_loss": float((detection_vector.detach().sum() / batch_size).cpu()),
            "kd_loss": float(kd_loss.detach().cpu()),
            "total_loss": float((vector.detach().sum() / batch_size).cpu()),
            "rm_loss": float(rm_loss.detach().cpu()),
            "rm_cls_loss": float(cls_loss.detach().cpu()),
            "rm_iou_loss": float(iou_loss.detach().cpu()),
            **counts,
        }
        return vector, loss_items

    @property
    def criterion(self):
        return getattr(self.student_model, "criterion", None)

    @criterion.setter
    def criterion(self, value):
        self.student_model.criterion = value

    def init_criterion(self):
        return self.student_model.init_criterion()

    @property
    def end2end(self):
        return self.student_model.end2end

    @end2end.setter
    def end2end(self, value):
        self.student_model.end2end = value

    def set_head_attr(self, **kwargs):
        self.student_model.set_head_attr(**kwargs)

    def fuse(self, *args, **kwargs):
        return self.student_model.fuse(*args, **kwargs)


KDModel = RMModel


## 5. Trainer dan checkpoint
EMA, validasi, dan `best.pt` memakai student biasa. State training tambahan disimpan terpisah; resume belum diverifikasi.

In [ ]:
def teacher_digest(model):
    digest = hashlib.sha256()
    for name, value in model.state_dict().items():
        digest.update(name.encode())
        digest.update(value.detach().cpu().contiguous().reshape(-1).view(torch.uint8).numpy().tobytes())
    return digest.hexdigest()

def gradient_norm(parameters):
    values = [p.grad.detach().float().norm() for p in parameters if p.grad is not None]
    result = float(torch.stack(values).norm()) if values else 0.0
    if not math.isfinite(result) or result <= 0:
        raise RuntimeError("Gradien harus finite dan nonzero.")
    return result

class StudentEMA(ModelEMA):
    def __init__(self, model):
        super().__init__(unwrap_model(model).student_model)
    def update(self, model):
        super().update(unwrap_model(model).student_model)
    def update_attr(self, model, include=(), exclude=()):
        super().update_attr(unwrap_model(model).student_model, include, exclude)

class ResearchTrainer(DetectionTrainer):
    def set_model_attributes(self):
        super().set_model_attributes()
        if self.args.resume or self.args.compile or self.world_size > 1:
            raise ValueError("Gunakan satu GPU, compile=False, resume=False; RESUME_NOT_VERIFIED.")
        teacher = YOLO(str(TEACHER_PATH)).model.to(self.device).eval().requires_grad_(False)
        self.model = KDModel(self.model, teacher, **METHOD_KWARGS).to(self.device)

    def build_optimizer(self, model, *args, **kwargs):
        model.teacher_model.eval().requires_grad_(False)
        optimizer = super().build_optimizer(model, *args, **kwargs)
        for group in optimizer.param_groups:
            group["params"] = [p for p in group["params"] if p.requires_grad]
        actual = [id(p) for group in optimizer.param_groups for p in group["params"]]
        expected = {id(p) for p in model.parameters() if p.requires_grad}
        if set(actual) != expected or len(actual) != len(set(actual)):
            raise RuntimeError("Parameter optimizer tidak sesuai.")
        return optimizer

    def _setup_train(self):
        super()._setup_train()
        self.ema = None
        self.ema = StudentEMA(self.model)

    def _build_train_pipeline(self):
        if self.batch_size != BATCH:
            raise RuntimeError("Batch berubah setelah OOM; hentikan dan tinjau konfigurasi.")
        super()._build_train_pipeline()

    def preprocess_batch(self, batch):
        batch = super().preprocess_batch(batch)
        self.actual_shape = tuple(batch["img"].shape)
        return batch

    def _handle_nan_recovery(self, epoch):
        if not torch.isfinite(self.loss).all() or not math.isfinite(float(self.fitness)):
            raise RuntimeError("Loss/fitness non-finite; training dihentikan.")
        return False

    def save_model(self):
        # Hanya DetectionModel biasa yang masuk best.pt/last.pt.
        student = copy.deepcopy(self.ema.ema).half().to(memory_format=torch.contiguous_format)
        student.criterion = None
        if not all(torch.isfinite(v).all() for v in student.state_dict().values() if v.is_floating_point()):
            raise RuntimeError("Checkpoint student mengandung NaN/Inf.")
        checkpoint = {"model": student, "ema": None, "optimizer": None, "epoch": int(self.epoch),
                      "best_fitness": float(self.best_fitness), "updates": self.ema.updates,
                      "train_args": vars(self.args), "train_metrics": {**self.metrics, "fitness": self.fitness},
                      "train_results": self.read_results_csv(), "version": metadata.version("ultralytics"),
                      "method": METHOD, "teacher_provenance": TEACHER_INFO, "resume_status": "INFERENCE_ONLY"}
        self.wdir.mkdir(parents=True, exist_ok=True)
        torch.save(checkpoint, self.last)
        if self.fitness == self.best_fitness:
            torch.save(checkpoint, self.best)
        return True


## 6. Logging W&B
Loss per komponen, mAP, precision/recall, LR, gradien, VRAM, dan durasi dicatat. Snapshot setiap epoch baru dibersihkan setelah upload selesai.

In [ ]:
def cpu_state(state):
    return {k: v.detach().cpu().clone() for k, v in state.items()}

def make_callbacks(run, diagnostic):
    context = {"steps": 0, "optimizer_steps": 0, "best_epoch": None, "sums": {}, "batches": 0}

    def start(trainer):
        context["teacher_before"] = teacher_digest(trainer.model.teacher_model)
        context["started"] = time.perf_counter()
        if trainer.device.type == "cuda":
            torch.cuda.reset_peak_memory_stats(trainer.device)
        def before_step(optimizer, *_):
            # Fused AdamW memanggil hook juga saat GradScaler melewati update.
            found_inf = getattr(optimizer, "found_inf", None)
            if found_inf is not None and float(found_inf) != 0:
                context["amp_skipped_steps"] = context.get("amp_skipped_steps", 0) + 1
                return
            context["student_grad"] = gradient_norm(trainer.model.student_model.parameters())
            if hasattr(trainer.model, "auxiliary_parameters"):
                context["aux_grad"] = gradient_norm(trainer.model.auxiliary_parameters())
            context["optimizer_steps"] += 1
        context["gradient_hook"] = trainer.optimizer.register_step_pre_hook(before_step)

    def batch_start(trainer):
        trainer._oom_retries = 3  # Tidak mengizinkan native OOM batch-halving.
        if trainer.batch_size != BATCH or int(trainer.args.imgsz) != IMGSZ or bool(trainer.amp) != AMP:
            raise RuntimeError("Batch, resolusi, atau AMP efektif berbeda dari konfigurasi.")

    def batch_end(trainer):
        context["steps"] += 1
        context["batches"] += 1
        for key, value in trainer.model.last_metrics.items():
            if not math.isfinite(float(value)):
                raise RuntimeError(f"Metrik non-finite: {key}")
            context["sums"][key] = context["sums"].get(key, 0) + float(value)
        teacher = trainer.model.teacher_model
        if teacher.training or any(p.grad is not None for p in teacher.parameters()):
            raise RuntimeError("Teacher keluar dari eval atau menerima gradien.")

    def save(trainer):
        epoch = int(trainer.epoch) + 1
        if trainer.fitness == trainer.best_fitness:
            context["best_epoch"] = epoch
        state = {"epoch": epoch, "student": cpu_state(trainer.model.student_model.state_dict()),
                 "auxiliary": cpu_state({k: v for k, v in trainer.model.state_dict().items()
                                         if not k.startswith(("student_model.", "teacher_model."))}),
                 "optimizer": trainer.optimizer.state_dict(), "scaler": trainer.scaler.state_dict(),
                 "resume_status": "RESUME_NOT_VERIFIED"}
        torch.save(state, trainer.wdir / "training_state_last.pt")
        if trainer.fitness == trainer.best_fitness:
            shutil.copy2(trainer.wdir / "training_state_last.pt", trainer.wdir / "training_state_best.pt")
        # Snapshot tetap utuh sampai upload selesai, lalu file sementara dibersihkan.
        with tempfile.TemporaryDirectory(prefix=f"epoch-{epoch:03d}-", dir=str(trainer.save_dir)) as temporary:
            snapshot = Path(temporary)
            item = wandb.Artifact(f"{EXPERIMENT_ID}-{run.id}-checkpoints", type="model",
                    metadata={"epoch": epoch, "best_epoch": context["best_epoch"], "method": METHOD,
                              "diagnostic": diagnostic, "teacher_sha256": TEACHER_SHA256,
                              "resume_status": "RESUME_NOT_VERIFIED"})
            for filename in ("last.pt", "best.pt", "training_state_last.pt", "training_state_best.pt"):
                shutil.copy2(trainer.wdir / filename, snapshot / filename)
                item.add_file(str(snapshot / filename), name=filename)
            run.log_artifact(item, aliases=["latest", f"epoch-{epoch:03d}"] +
                             (["best"] if trainer.fitness == trainer.best_fitness else [])).wait()
        values = {f"train/{k}": v / context["batches"] for k, v in context["sums"].items()}
        values.update(trainer.metrics)
        values.update(trainer.lr)
        values.update(epoch=epoch, best_epoch=context["best_epoch"],
                      elapsed_seconds=time.perf_counter() - context["started"],
                      amp_skipped_steps=context.get("amp_skipped_steps", 0),
                      student_gradient_norm=context.get("student_grad", 0),
                      auxiliary_gradient_norm=context.get("aux_grad", 0))
        if trainer.device.type == "cuda":
            values.update(peak_allocated_gib=torch.cuda.max_memory_allocated() / 2**30,
                          peak_reserved_gib=torch.cuda.max_memory_reserved() / 2**30)
        run.log(values, step=epoch)
        context["sums"], context["batches"] = {}, 0

    return {"on_train_start": start, "on_train_batch_start": batch_start,
            "on_train_batch_end": batch_end, "on_model_save": save}, context


In [ ]:
def experiment_config():
    return {"method": METHOD, "model": MODEL_WEIGHTS, "data": DATA, "epochs": EPOCHS,
            "imgsz": IMGSZ, "batch": BATCH, "seed": SEED, "optimizer": OPTIMIZER, "lr0": LR,
            "weight_decay": WEIGHT_DECAY, "warmup_epochs": WARMUP_EPOCHS,
            "close_mosaic": CLOSE_MOSAIC, "amp": AMP, "workers": WORKERS,
            "method_kwargs": METHOD_KWARGS, "teacher_sha256": TEACHER_SHA256}

def run_training(diagnostic=False):
    if not 1 <= EPOCHS <= 100:
        raise ValueError("Gunakan 1 sampai 100 epoch.")
    config = experiment_config()
    name = ("diagnostic" if diagnostic else "training") + "_" + uuid.uuid4().hex[:6]
    run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, group=WANDB_GROUP,
                     name=f"{EXPERIMENT_ID}_seed{SEED}_{name}", id=uuid.uuid4().hex[:8], resume="never",
                     tags=[*([METHOD, "teacher-s", "student-n", f"seed{SEED}", "poltrada-v3", name.split("_")[0]]), "colab"],
                     config={**({**config, "dataset": DATASET_INFO, "teacher": TEACHER_INFO}), "colab_dataset": COLAB_DATA_PROVENANCE, "runtime": "google-colab"}, dir=str(SESSION_DIR))
    callbacks, context = make_callbacks(run, diagnostic)
    trainer = None
    try:
        trainer = ResearchTrainer(overrides={"model": MODEL_WEIGHTS, "data": DATA, "epochs": 1 if diagnostic else EPOCHS,
            "imgsz": IMGSZ, "batch": BATCH, "seed": SEED, "device": DEVICE, "workers": WORKERS,
            "optimizer": OPTIMIZER, "lr0": LR, "weight_decay": WEIGHT_DECAY,
            "warmup_epochs": WARMUP_EPOCHS, "close_mosaic": CLOSE_MOSAIC, "amp": AMP,
            "patience": 100, "split": EVAL_SPLIT, "project": str(SESSION_DIR), "name": name,
            "plots": not diagnostic, "save": True, "exist_ok": False, "resume": False,
            "compile": False, "distill_model": None})
        for event, callback in callbacks.items():
            trainer.add_callback(event, callback)
        trainer.train()
        if context["optimizer_steps"] < (2 if diagnostic else 1):
            raise RuntimeError("Belum ada cukup optimizer step nyata yang berhasil.")
        if teacher_digest(trainer.model.teacher_model) != context["teacher_before"]:
            raise RuntimeError("Parameter atau BatchNorm teacher berubah.")
        if not all(math.isfinite(float(v)) for v in trainer.metrics.values()):
            raise RuntimeError("Metrik validasi non-finite.")
        student = YOLO(str(trainer.best)).model.eval()
        if student.yaml.get("scale") != "n" or hasattr(student, "teacher_model"):
            raise RuntimeError("Checkpoint inferensi bukan student-only.")
        result = {"status": "PASS", "config": config, "epochs_this_run": 1 if diagnostic else EPOCHS,
                  "training_batches": context["steps"], "best_pt": str(trainer.best),
                  "best_epoch": context["best_epoch"], "metrics": trainer.metrics,
                  "optimizer_steps": context["optimizer_steps"],
                  "seconds": time.perf_counter() - context["started"], "resume": "RESUME_NOT_VERIFIED"}
        (Path(trainer.save_dir) / "summary.json").write_text(json.dumps(result, indent=2))
        run.summary.update(result)
        run.finish(exit_code=0)
        return result
    except BaseException:
        run.finish(exit_code=1)
        raise
    finally:
        if context.get("gradient_hook"):
            context["gradient_hook"].remove()


## 7. Verifikasi numerik dan smoke test opsional
Default `RUN_SMOKE_TEST=False`: tidak ada smoke run tambahan. Jika diaktifkan, satu epoch pada seluruh split train, lalu validasi dan load ulang student. Jika OOM/AMP/loss gagal, training berhenti tanpa menurunkan batch atau resolusi otomatis.

In [ ]:
def rm_numerical_verification():
    """Independent scalar oracle plus reversed-order and student-gradient checks."""
    teacher_values = [0.92, 0.53, 0.08]
    student_values = [0.08, 0.53, 0.92]
    def scalar_softmax(values):
        shifted = [math.exp(value - max(values)) for value in values]
        return [value / sum(shifted) for value in shifted]
    pt, ps = scalar_softmax(teacher_values), scalar_softmax(student_values)
    oracle = sum(t * math.log(t / s) for t, s in zip(pt, ps))
    teacher = torch.tensor(teacher_values, dtype=torch.float64, requires_grad=True)
    reversed_student = torch.tensor(student_values, dtype=torch.float64, requires_grad=True)
    actual = rm_rank_kl(reversed_student, teacher)
    matched = rm_rank_kl(teacher.detach(), teacher)
    actual.backward()
    if not math.isclose(float(actual.detach()), oracle, abs_tol=1e-12, rel_tol=1e-12):
        raise AssertionError("Candidate-axis KL disagrees with independent scalar equation (6) oracle")
    if not float(actual.detach()) > float(matched):
        raise AssertionError("Reversed student ranking must have larger KL than matched ranking")
    if not torch.isfinite(reversed_student.grad).all() or reversed_student.grad.abs().sum() == 0:
        raise AssertionError("RM student gradient must be finite and nonzero")
    if teacher.grad is not None:
        raise AssertionError("Teacher rank targets must not receive gradients")
    return {"status": "PASS CPU", "oracle": oracle, "actual": float(actual.detach()),
            "matched_loss": float(matched), "reversed_loss": float(actual.detach()),
            "student_gradient_l1": float(reversed_student.grad.abs().sum()),
            "reference": "AAAI 2022 equations 5/6; independent Python math oracle"}

print(rm_numerical_verification())


In [ ]:
DIAGNOSTIC_RESULT = None
if RUN_SMOKE_TEST:
    DIAGNOSTIC_RESULT = run_training(diagnostic=True)
    print(json.dumps(DIAGNOSTIC_RESULT, indent=2))
else:
    print('Smoke test dilewati; konfigurasi langsung training.')


## 8. Training dan hasil
Satu metode, satu seed, maksimal 100 epoch. Untuk uji satu epoch saja, set `EPOCHS=1`.

In [ ]:
TRAINING_RESULT = None
if RUN_FULL_TRAINING:
    if RUN_SMOKE_TEST and (not DIAGNOSTIC_RESULT or DIAGNOSTIC_RESULT["status"] != "PASS" or DIAGNOSTIC_RESULT["config"] != experiment_config()):
        raise RuntimeError("Jalankan ulang smoke test setelah mengubah konfigurasi.")
    TRAINING_RESULT = run_training(diagnostic=False)
else:
    print("Training dinonaktifkan: RUN_FULL_TRAINING=False.")


In [ ]:
result = TRAINING_RESULT or DIAGNOSTIC_RESULT
if result is not None:
    print("Checkpoint student:", result["best_pt"])
    print("Best epoch:", result["best_epoch"])
    print("Validation:", result["metrics"])
    print("Durasi (detik):", round(result["seconds"], 2))
else:
    print("Belum ada hasil training atau smoke test.")
